# CropCare — confidence threshold calibration

Answers one question with a measurement instead of a guess: **at what
confidence should the app say "I'm not sure" instead of naming a disease?**

`confidenceThreshold` in `ml_inference_service.dart` is currently **0.70,
reasoned but not measured** — see the comment above it in that file. This
notebook measures it against your actual trained model's actual predictions on
field photographs it never trained on, and prints the number to replace it
with.

## Before you run

1. **Settings → Accelerator → None.** This is CPU-only inference over a few
   hundred images — a GPU buys you nothing here and just costs you queue time.
2. **Internet: On.**
3. **Add Input:**
   - Your trained model. If it isn't already a Kaggle Dataset, create one:
     **Datasets → New Dataset**, upload the `.tflite` file you downloaded from
     the training run, give it any name. Then attach it here like any other
     dataset.
   - `plantdoc` (Datasets tab) — the same held-out field test set the training
     notebook used. Same caveat as there: if you attach a *notebook* instead
     of the *dataset*, this will find nothing — see the training notebook's
     own note on that mistake if you hit it again.

Missing PlantDoc means this notebook has nothing to measure against and will
stop with a clear message rather than fabricating a number.

## 1 · Setup

In [ ]:
import os, re, collections
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

print("TensorFlow", tf.__version__)
print("This notebook is CPU-only by design - no GPU check needed.")

## 2 · Taxonomy (same source of truth as training)

In [ ]:
"""
CropCare model taxonomy: the canonical class list, and how each source
dataset's labels map onto it.

This file is the single source of truth. The training notebook builds its label
space from it, the evaluation harness reports against it, and `emit_dart.py`
generates the Dart class list and disease-id map from it. Change a class here
and everything downstream follows.

-----------------------------------------------------------------------------
Why the class list changed
-----------------------------------------------------------------------------
The shipped model is stock PlantVillage: 38 classes, of which 24 are apple,
blueberry, cherry, grape, orange, peach, raspberry, soybean, squash and
strawberry - temperate crops a Sri Lankan smallholder will never photograph.
It has no rice at all, despite rice being the staple crop and despite the app
already seeding `paddy` with disease rows and a translated treatment guideline.
It has exactly one arthropod class, so pests are effectively unsupported.

The taxonomy below drops the temperate fruit entirely and adds rice and
cassava. The count barely changes; the relevance changes completely.

-----------------------------------------------------------------------------
Why the training data changed
-----------------------------------------------------------------------------
PlantVillage is lab photography - detached leaves on uniform backgrounds. A
model scoring 99.35% on its own test split drops to 31.4% on field images, and
a classifier trained on 8 background pixels alone reaches 49% accuracy, which
means the network is substantially reading the backdrop rather than the leaf.

So PlantVillage is kept only as one source among several, and the field
datasets carry the real weight. PlantDoc is held out of training entirely and
used as the field test set - see `HELD_OUT_SOURCES`. A number that is not
measured on unseen field photographs is not worth reporting.
"""


from dataclasses import dataclass, field


# ---------------------------------------------------------------------------
# Canonical classes
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class CropClass:
    """One output class of the model."""

    # Matches the app's `disease` table id convention: {crop}_{condition}.
    id: str
    crop_id: str
    name_en: str

    # 'low' | 'moderate' | 'high' | None. Mirrors disease.severity_default.
    severity: str | None = None

    # True for arthropod damage rather than pathogen infection. The app can
    # word these differently: "pest" and "disease" call for different action,
    # and lumping them under one label was part of why pests went unserved.
    is_pest: bool = False

    healthy: bool = False


def _c(**kw) -> CropClass:
    return CropClass(**kw)


# Ordered. The index of each entry IS the model's output index, so appending
# is safe and reordering is not. `emit_dart.py` relies on this.
CLASSES: list[CropClass] = [
    # -- Rice / paddy ------------------------------------------------------
    # The staple crop, and the app's largest gap: `paddy` is a seeded crop
    # with disease rows and a translated guideline, but the current model
    # cannot predict it at all, so a rice photo returns a confident tomato
    # answer.
    _c(id="paddy_bacterial_leaf_blight", crop_id="paddy",
       name_en="Bacterial Leaf Blight", severity="high"),
    _c(id="paddy_bacterial_leaf_streak", crop_id="paddy",
       name_en="Bacterial Leaf Streak", severity="moderate"),
    _c(id="paddy_bacterial_panicle_blight", crop_id="paddy",
       name_en="Bacterial Panicle Blight", severity="high"),
    _c(id="paddy_blast", crop_id="paddy",
       name_en="Rice Blast", severity="high"),
    _c(id="paddy_brown_spot", crop_id="paddy",
       name_en="Brown Spot", severity="moderate"),
    _c(id="paddy_downy_mildew", crop_id="paddy",
       name_en="Downy Mildew", severity="moderate"),
    _c(id="paddy_tungro", crop_id="paddy",
       name_en="Tungro Virus", severity="high"),
    _c(id="paddy_dead_heart", crop_id="paddy",
       name_en="Stem Borer (Dead Heart)", severity="high", is_pest=True),
    _c(id="paddy_hispa", crop_id="paddy",
       name_en="Rice Hispa", severity="moderate", is_pest=True),
    _c(id="paddy_healthy", crop_id="paddy",
       name_en="Healthy", healthy=True),

    # -- Tomato ------------------------------------------------------------
    _c(id="tomato_bacterial_spot", crop_id="tomato",
       name_en="Bacterial Spot", severity="moderate"),
    _c(id="tomato_early_blight", crop_id="tomato",
       name_en="Early Blight", severity="moderate"),
    _c(id="tomato_late_blight", crop_id="tomato",
       name_en="Late Blight", severity="high"),
    _c(id="tomato_leaf_mold", crop_id="tomato",
       name_en="Leaf Mold", severity="moderate"),
    _c(id="tomato_septoria_leaf_spot", crop_id="tomato",
       name_en="Septoria Leaf Spot", severity="moderate"),
    _c(id="tomato_target_spot", crop_id="tomato",
       name_en="Target Spot", severity="moderate"),
    _c(id="tomato_yellow_leaf_curl_virus", crop_id="tomato",
       name_en="Yellow Leaf Curl Virus", severity="high"),
    _c(id="tomato_mosaic_virus", crop_id="tomato",
       name_en="Mosaic Virus", severity="high"),
    _c(id="tomato_spider_mites", crop_id="tomato",
       name_en="Two-Spotted Spider Mite", severity="moderate", is_pest=True),
    _c(id="tomato_healthy", crop_id="tomato", name_en="Healthy", healthy=True),

    # -- Chili / pepper ----------------------------------------------------
    # PlantVillage's "Pepper,_bell" is the closest available proxy. Noted in
    # the app already (TD-006); it stays a proxy, not a claim of equivalence.
    _c(id="chili_bacterial_spot", crop_id="chili",
       name_en="Bacterial Spot", severity="moderate"),
    _c(id="chili_healthy", crop_id="chili", name_en="Healthy", healthy=True),

    # -- Potato ------------------------------------------------------------
    _c(id="potato_early_blight", crop_id="potato",
       name_en="Early Blight", severity="moderate"),
    _c(id="potato_late_blight", crop_id="potato",
       name_en="Late Blight", severity="high"),
    _c(id="potato_healthy", crop_id="potato", name_en="Healthy", healthy=True),

    # -- Cassava / manioc --------------------------------------------------
    # Widely grown by Sri Lankan smallholders, and the dataset is genuine
    # field survey photography rather than lab plates.
    _c(id="cassava_bacterial_blight", crop_id="cassava",
       name_en="Cassava Bacterial Blight", severity="high"),
    _c(id="cassava_brown_streak", crop_id="cassava",
       name_en="Cassava Brown Streak Disease", severity="high"),
    _c(id="cassava_green_mottle", crop_id="cassava",
       name_en="Cassava Green Mottle", severity="moderate"),
    _c(id="cassava_mosaic", crop_id="cassava",
       name_en="Cassava Mosaic Disease", severity="high"),
    _c(id="cassava_healthy", crop_id="cassava",
       name_en="Healthy", healthy=True),

    # -- Maize -------------------------------------------------------------
    _c(id="corn_gray_leaf_spot", crop_id="corn",
       name_en="Gray Leaf Spot", severity="moderate"),
    _c(id="corn_common_rust", crop_id="corn",
       name_en="Common Rust", severity="moderate"),
    _c(id="corn_northern_leaf_blight", crop_id="corn",
       name_en="Northern Leaf Blight", severity="moderate"),
    _c(id="corn_healthy", crop_id="corn", name_en="Healthy", healthy=True),
]

CLASS_IDS: list[str] = [c.id for c in CLASSES]
CLASS_INDEX: dict[str, int] = {c.id: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

CROPS: list[str] = sorted({c.crop_id for c in CLASSES})


# ---------------------------------------------------------------------------
# Source datasets
# ---------------------------------------------------------------------------
@dataclass
class SourceDataset:
    """A dataset to draw training images from."""

    key: str
    name: str

    # Substrings matched (case-insensitively) against directory names under
    # /kaggle/input, so the notebook works regardless of which mirror of a
    # dataset the user attaches. Exact slugs vary between mirrors and go stale;
    # discovery does not.
    dir_hints: list[str]

    # Maps a source label (a folder name, or a value from a CSV) onto a
    # canonical class id. Anything unmapped is REPORTED, never silently
    # dropped - a quietly discarded third of a dataset is the kind of bug that
    # only shows up as unexplained accuracy loss.
    label_map: dict[str, str] = field(default_factory=dict)

    # Some datasets label via a CSV rather than folder names.
    csv_name: str | None = None
    csv_image_col: str | None = None
    csv_label_col: str | None = None

    # Cassava ships integer labels plus a JSON legend.
    csv_label_is_int: bool = False

    notes: str = ""


PADDY_DOCTOR = SourceDataset(
    key="paddy_doctor",
    name="Paddy Doctor (rice, field)",
    dir_hints=["paddy-disease-classification", "paddy_doctor", "paddy-doctor"],
    csv_name="train.csv",
    csv_image_col="image_id",
    csv_label_col="label",
    notes="Field photography of rice. Closes the app's paddy gap. Two of its "
          "classes (dead_heart, hispa) are insect damage, which gives real "
          "pest coverage for the staple crop without needing IP102.",
    label_map={
        "bacterial_leaf_blight": "paddy_bacterial_leaf_blight",
        "bacterial_leaf_streak": "paddy_bacterial_leaf_streak",
        "bacterial_panicle_blight": "paddy_bacterial_panicle_blight",
        "blast": "paddy_blast",
        "brown_spot": "paddy_brown_spot",
        "downy_mildew": "paddy_downy_mildew",
        "tungro": "paddy_tungro",
        "dead_heart": "paddy_dead_heart",
        "hispa": "paddy_hispa",
        "normal": "paddy_healthy",
    },
)

CASSAVA = SourceDataset(
    key="cassava",
    name="Cassava Leaf Disease (field survey)",
    dir_hints=["cassava-leaf-disease-classification", "cassava"],
    csv_name="train.csv",
    csv_image_col="image_id",
    csv_label_col="label",
    csv_label_is_int=True,
    notes="Collected during a field survey in Uganda; genuinely in-the-wild.",
    label_map={
        "0": "cassava_bacterial_blight",
        "1": "cassava_brown_streak",
        "2": "cassava_green_mottle",
        "3": "cassava_mosaic",
        "4": "cassava_healthy",
    },
)

PLANT_VILLAGE = SourceDataset(
    key="plantvillage",
    name="PlantVillage (lab)",
    dir_hints=["plantvillage", "new-plant-diseases", "plant-village",
               "plant_village"],
    notes="Lab plates on uniform backgrounds. Kept ONLY as extra signal for "
          "classes the field datasets cover thinly. Never the sole source for "
          "a class, and never the test set - see the background-bias note at "
          "the top of this file.",
    label_map={
        "Tomato___Bacterial_spot": "tomato_bacterial_spot",
        "Tomato___Early_blight": "tomato_early_blight",
        "Tomato___Late_blight": "tomato_late_blight",
        "Tomato___Leaf_Mold": "tomato_leaf_mold",
        "Tomato___Septoria_leaf_spot": "tomato_septoria_leaf_spot",
        "Tomato___Target_Spot": "tomato_target_spot",
        "Tomato___Tomato_Yellow_Leaf_Curl_Virus":
            "tomato_yellow_leaf_curl_virus",
        "Tomato___Tomato_mosaic_virus": "tomato_mosaic_virus",
        "Tomato___Spider_mites Two-spotted_spider_mite": "tomato_spider_mites",
        "Tomato___healthy": "tomato_healthy",
        "Pepper,_bell___Bacterial_spot": "chili_bacterial_spot",
        "Pepper,_bell___healthy": "chili_healthy",
        "Potato___Early_blight": "potato_early_blight",
        "Potato___Late_blight": "potato_late_blight",
        "Potato___healthy": "potato_healthy",
        "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot":
            "corn_gray_leaf_spot",
        "Corn_(maize)___Common_rust_": "corn_common_rust",
        "Corn_(maize)___Northern_Leaf_Blight": "corn_northern_leaf_blight",
        "Corn_(maize)___healthy": "corn_healthy",
        # Everything else in PlantVillage - apple, blueberry, cherry, grape,
        # orange, peach, raspberry, soybean, squash, strawberry - is
        # deliberately absent. Those crops are not grown by the users this app
        # is for, and carrying them costs accuracy on the ones that are.
    },
)

PLANT_DOC = SourceDataset(
    key="plantdoc",
    name="PlantDoc (field, HELD OUT)",
    dir_hints=["plantdoc", "plant-doc", "plant_doc"],
    notes="Small, in-the-wild, scraped from the internet. Held out of "
          "training entirely and used as the field test set. This is the "
          "number that actually predicts how the app behaves in a field.",
    label_map={
        # PlantDoc folder naming differs between mirrors, so several spellings
        # map to the same class. Unmapped folders are reported at load time.
        "Tomato leaf bacterial spot": "tomato_bacterial_spot",
        "Tomato Early blight leaf": "tomato_early_blight",
        "Tomato leaf late blight": "tomato_late_blight",
        "Tomato leaf mosaic virus": "tomato_mosaic_virus",
        "Tomato leaf yellow virus": "tomato_yellow_leaf_curl_virus",
        "Tomato Septoria leaf spot": "tomato_septoria_leaf_spot",
        "Tomato mold leaf": "tomato_leaf_mold",
        "Tomato two spotted spider mites leaf": "tomato_spider_mites",
        "Tomato leaf": "tomato_healthy",
        "Bell_pepper leaf spot": "chili_bacterial_spot",
        "Bell_pepper leaf": "chili_healthy",
        "Potato leaf early blight": "potato_early_blight",
        "Potato leaf late blight": "potato_late_blight",
        "Potato leaf": "potato_healthy",
        "Corn Gray leaf spot": "corn_gray_leaf_spot",
        "Corn rust leaf": "corn_common_rust",
        "Corn leaf blight": "corn_northern_leaf_blight",
    },
)

SOURCES: list[SourceDataset] = [PADDY_DOCTOR, CASSAVA, PLANT_VILLAGE, PLANT_DOC]

# Never trained on. Reported separately as the field generalisation number.
HELD_OUT_SOURCES = {"plantdoc"}

# Sources whose images are lab plates. Tracked so the notebook can report what
# fraction of each class's training data is lab rather than field - a class fed
# only by PlantVillage should be treated as unproven no matter what its
# validation accuracy says.
LAB_SOURCES = {"plantvillage"}


def summary() -> str:
    lines = [
        f"{NUM_CLASSES} classes across {len(CROPS)} crops: {', '.join(CROPS)}",
        f"{sum(1 for c in CLASSES if c.is_pest)} pest classes, "
        f"{sum(1 for c in CLASSES if c.healthy)} healthy classes",
        "",
        "Sources:",
    ]
    for s in SOURCES:
        held = " [HELD OUT - field test set]" if s.key in HELD_OUT_SOURCES else ""
        lines.append(f"  - {s.name}{held}: {len(s.label_map)} mapped labels")
    return "\n".join(lines)

print(summary())

## 3 · Find the model and PlantDoc

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
MAX_DEPTH = 3

def _norm(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())

def _candidate_dirs():
    if not INPUT_ROOT.exists():
        return []
    out = []
    for depth in range(1, MAX_DEPTH + 1):
        out.extend(sorted(p for p in INPUT_ROOT.glob("/".join(["*"] * depth))
                          if p.is_dir()))
    return out

def find_dirs(hints):
    hits = []
    for entry in _candidate_dirs():
        rel = _norm(str(entry.relative_to(INPUT_ROOT)))
        if not any(_norm(h) in rel for h in hints):
            continue
        if any(entry.is_relative_to(h) for h in hits):
            continue
        hits.append(entry)
    return hits

print("Mounted under /kaggle/input:")
if not INPUT_ROOT.exists() or not any(INPUT_ROOT.iterdir()):
    print("   (nothing)")
else:
    for top in sorted(p for p in INPUT_ROOT.iterdir() if p.is_dir()):
        print(f"   {top.name}/")

tflite_candidates = list(INPUT_ROOT.rglob("*.tflite"))
plantdoc_dirs = [d for d in find_dirs(PLANT_DOC.dir_hints)
                 if any(p.suffix.lower() in {".jpg",".jpeg",".png"}
                        for p in d.rglob("*"))]

if not tflite_candidates:
    raise SystemExit(
        "No .tflite file found under /kaggle/input. Upload it as a Kaggle "
        "Dataset (Datasets -> New Dataset -> upload the file) and attach it "
        "to this notebook, then Add Input again."
    )
if not plantdoc_dirs:
    raise SystemExit(
        "PlantDoc not found. Without it there is nothing to measure "
        "confidence against - see the note at the top of this notebook "
        "about attaching the dataset, not a notebook of the same name."
    )

MODEL_PATH = tflite_candidates[0]
PLANTDOC_DIR = plantdoc_dirs[0]
print(f"\nUsing model: {MODEL_PATH}")
print(f"Using PlantDoc: {PLANTDOC_DIR}")
if len(tflite_candidates) > 1:
    print(f"(found {len(tflite_candidates)} .tflite files, using the first - "
          f"remove the others from Input if this is not the right one)")

## 4 · Load PlantDoc labels, load the model

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
IMG_SIZE = 224

def load_plantdoc():
    norm = {re.sub(r"[^a-z0-9]", "", k.lower()): v
            for k, v in PLANT_DOC.label_map.items()}
    rows, unmapped = [], collections.Counter()
    for folder in sorted({p.parent for p in PLANTDOC_DIR.rglob("*")
                          if p.suffix in IMG_EXT}):
        key = re.sub(r"[^a-z0-9]", "", folder.name.lower())
        class_id = norm.get(key)
        if class_id is None:
            unmapped[folder.name] += sum(1 for p in folder.iterdir()
                                         if p.suffix in IMG_EXT)
            continue
        for img in folder.iterdir():
            if img.suffix in IMG_EXT:
                rows.append((str(img), class_id))
    if unmapped:
        print(f"{sum(unmapped.values())} images under unmapped labels, skipped "
              f"(expected - PlantDoc covers more crops than this taxonomy):")
        for label, n in unmapped.most_common(5):
            print(f"   {label}: {n}")
    return pd.DataFrame(rows, columns=["path", "class_id"])

df = load_plantdoc()
df["label"] = df["class_id"].map(CLASS_INDEX)
print(f"\n{len(df)} labelled field images across {df['class_id'].nunique()} classes")

interpreter = tf.lite.Interpreter(model_path=str(MODEL_PATH))
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]
print(f"\nmodel input  {inp['shape']} {inp['dtype'].__name__}")
print(f"model output {out['shape']} {out['dtype'].__name__}")
assert int(out["shape"][-1]) == NUM_CLASSES, (
    f"Model has {out['shape'][-1]} output classes but taxonomy.py defines "
    f"{NUM_CLASSES}. Wrong model file attached, or taxonomy has drifted - "
    f"stop and check before trusting anything below."
)

## 5 · Run inference over every field image

Reproduces the app's own preprocessing exactly: resize to 224x224, divide by
255. If this notebook's numbers do not match what the app actually shows on a
real photo, this is the first place to check for a mismatch.

In [ ]:
def preprocess(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return img.numpy()[None, ...].astype(inp["dtype"])

def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

results = []
for i, row in df.iterrows():
    try:
        batch = preprocess(row["path"])
    except Exception as exc:
        continue
    interpreter.set_tensor(inp["index"], batch)
    interpreter.invoke()
    logits = interpreter.get_tensor(out["index"])[0]
    probs = softmax(logits)
    pred_idx = int(np.argmax(probs))
    results.append({
        "true_class": row["class_id"],
        "true_idx": row["label"],
        "pred_class": CLASS_IDS[pred_idx],
        "pred_idx": pred_idx,
        "confidence": float(probs[pred_idx]),
        "correct": pred_idx == row["label"],
    })
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(df)}")

res = pd.DataFrame(results)
print(f"\nRan inference on {len(res)} images. Overall top-1 accuracy: "
      f"{res['correct'].mean():.1%}")

## 6 · The table that actually matters

For each candidate threshold: if the app only trusted predictions AT OR ABOVE
it, what fraction of those trusted predictions would be correct
(**precision**), and what fraction of all correct diagnoses would still get
shown as confident (**coverage**)?

There is no single "right" answer here - it is a real product tradeoff.
A high threshold means the app says "not sure" more often but is rarely wrong
when it does commit. A low threshold means it commits more often but is wrong
more often too. Pick the row whose tradeoff you're willing to stand behind.

In [ ]:
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

print(f"{'threshold':>9}  {'n trusted':>9}  {'precision':>9}  {'coverage':>9}")
rows = []
for t in thresholds:
    trusted = res[res["confidence"] >= t]
    precision = trusted["correct"].mean() if len(trusted) else float("nan")
    coverage = trusted["correct"].sum() / max(1, res["correct"].sum())
    rows.append((t, len(trusted), precision, coverage))
    print(f"{t:>9.2f}  {len(trusted):>9d}  {precision:>9.1%}  {coverage:>9.1%}")

rows_df = pd.DataFrame(rows, columns=["threshold", "n_trusted", "precision", "coverage"])

# A reasoned starting suggestion, not a mandate: the lowest threshold that
# still keeps precision at or above 85%. Below 0.30 or when nothing clears
# 85% precision at all, this intentionally does not guess - look at the full
# table above instead.
candidates = rows_df[rows_df["precision"] >= 0.85]
if len(candidates):
    suggested = candidates["threshold"].min()
    print(f"\nSuggested confidenceThreshold: {suggested:.2f}")
    print("(lowest threshold that keeps precision at or above 85% on this "
          "field test set - reasoning, not a mandate; use the table above "
          "to pick a different tradeoff if 85% is not the bar you want)")
else:
    print("\nNo threshold in the tested range reaches 85% precision on this "
          "field test set. Do not round up to a number that sounds safe - "
          "this is telling you the model needs more work before the app "
          "should lean on confidence alone. Report this rather than picking "
          "a threshold that hides it.")

## 7 · Where it's actually going wrong

The threshold is a blunt instrument if one or two classes are dragging the
whole number down. Worth knowing which ones before you finalize anything.

In [ ]:
per_class = (res.groupby("true_class")
             .agg(n=("correct", "size"), acc=("correct", "mean"),
                  avg_confidence=("confidence", "mean"))
             .sort_values("acc"))
print(f"{'class':<38} {'n':>4} {'acc':>6} {'avg conf':>9}")
for cls, row in per_class.iterrows():
    flag = "  <-- weak" if row["acc"] < 0.5 else ""
    print(f"{cls:<38} {row['n']:>4.0f} {row['acc']:>6.2f} {row['avg_confidence']:>9.2f}{flag}")

## 8 · Apply it

Open `lib/data/local/ml/ml_inference_service.dart` and replace the value of
`confidenceThreshold` with the number from section 6 (or whichever row from
the table you decided fits the tradeoff you want).

Update the comment above it too — replace "PROVISIONAL" with what this
notebook measured, the date, and the precision/coverage at that value, so the
next person (including future you) does not have to re-derive it. Something
like:

```
// Calibrated 2026-XX-XX against N field images (PlantDoc, held out from
// training). At this threshold: XX% precision, XX% coverage.
// See ml/build_calibration_notebook.py.
static const double confidenceThreshold = 0.XX;
```

No other file needs to change. `entropyThreshold` was a separate, deliberate
decision (see the comment above it) and this notebook does not touch it.